# 02 — Ablation Study

Три эксперимента поверх реального корпуса (Трудовой + Налоговый кодекс РК) и
25-вопросного ground truth (`evaluation/ground_truth.jsonl`):

- **A. chunk_size** — recursive-чанкинг {200,400,800,1600} против baseline
  `section` (article-aware).
- **B. retrieval** — vector vs hybrid (BM25 + RRF) при фиксированном section-чанкинге.
- **C. min_chunk** — влияние фильтрации чанков-пустышек из оглавления
  (`min_chunk_chars` 0 vs 120) на precision@5.

Вся логика — в `scripts/run_ablation.py` (переиспользует `build_chunks` из
`scripts/build_index.py`, `FaissStore`/`Retriever`/`build_embedder` из
`app/retrieval.py`, `evaluate`/`load_gt` из `evaluation/evaluate_retrieval.py`),
так что ноутбук и `python3 scripts/run_ablation.py --experiment ...` не могут разойтись.

Каждая конфигурация запускается в своём собственном subprocess и индексируется
в свой временный каталог `data/.ablation/<config>/` — рабочий индекс сервиса
(`data/index/`) не трогается. Изоляция по процессам — не архитектурная прихоть:
повторные тяжёлые вызовы torch/faiss в одном долгоживущем процессе падают с
segfault при завершении (конфликт OpenMP-рантаймов) под устойчивой нагрузкой на
этой машине; на отдельный процесс — только уже посчитанный конфиг теряется, а не
весь эксперимент. Эмбеддинг-кэш на диске (`data/.embed_cache/`) общий для всех
подпроцессов, поэтому повторный запуск этого ноутбука — быстрый (cache hit), а
первый прогон нового чанк-сайза — нет.

> `EMBEDDING_BACKEND=st` (реальная Sentence-BERT модель, реальный корпус). Для
> офлайн/CI смысла нет — гейт `recall@5 >= 0.75` из `tests/test_retrieval.py`
> проверяется на фикстуре с hash-эмбеддером, а не на этих числах.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "evaluation"))

from run_ablation import (
    experiment_chunk_size,
    experiment_retrieval,
    experiment_min_chunk,
    write_json,
    write_markdown,
    write_plot,
    _TITLES,
    GT_PATH,
)
from evaluate_retrieval import load_gt
from IPython.display import Image, Markdown, display

RESULTS = ROOT / "evaluation" / "results"
TOP_K = 5


def show(rows, experiment):
    """Write json/md/png for one experiment's rows and render them inline."""
    json_path = RESULTS / f"ablation_{experiment}.json"
    md_path = RESULTS / f"ablation_{experiment}.md"
    png_path = RESULTS / f"ablation_{experiment}.png"
    write_json(rows, json_path, experiment=experiment)
    write_markdown(rows, md_path, experiment=experiment, title=_TITLES[experiment], top_k=TOP_K)
    write_plot(rows, png_path, experiment=experiment, top_k=TOP_K)
    display(Markdown(md_path.read_text(encoding="utf-8")))
    display(Image(filename=str(png_path)))


gt = load_gt(str(GT_PATH))
print(f"ground_truth_questions={len(gt)}")

## A. Chunk-size sweep (recursive) vs section baseline

In [ ]:
rows_chunk_size = experiment_chunk_size(top_k=TOP_K)
show(rows_chunk_size, "chunk_size")

## B. Retrieval strategy: vector vs hybrid (section chunking)

In [ ]:
rows_retrieval = experiment_retrieval(top_k=TOP_K)
show(rows_retrieval, "retrieval")

## C. min_chunk_chars: filtering table-of-contents stubs

In [ ]:
rows_min_chunk = experiment_min_chunk(top_k=TOP_K)
show(rows_min_chunk, "min_chunk")

## Вывод (заполнить по своим числам)

> Пример формата: «section даёт recall@5=X.XX против Y.YY у recursive-800, потому
> что статьи самодостаточны и не рвутся между чанками; hybrid добавляет +Z за счёт
> точного лексического матча номеров статей; фильтрация min_chunk_chars=120 поднимает
> recall с A.AA до B.BB, но снижает precision@5 с C.CC до D.DD — объяснить почему.»